# Complete Evaluation Notebook — Java Vulnerability Fine-Tuned Model

**AMD MI300X (192GB VRAM) — Native 16-bit Inference**

This notebook loads your saved `large-java-vuln-adapter` and evaluates it against the **test set** (data the model has NEVER seen during training).

In [ ]:
# Step 1: Install all evaluation dependencies
!pip install -q evaluate rouge_score bert_score codebleu

In [ ]:
# Step 2: Load model in native 16-bit (No BitsAndBytes - avoids ROCm crashes!)
import sys
import torch
from unittest.mock import MagicMock
sys.modules['transformers.integrations.deepgemm'] = MagicMock()
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

model_id = "Qwen/Qwen2.5-Coder-32B-Instruct"
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print("Loading Base 32B Model in Native bfloat16 (~64GB VRAM)...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print("Applying Your Custom Adapter Weights...")
model = PeftModel.from_pretrained(base_model, "./java-vuln-adapter-32b-full")
model.config.use_cache = True
model.eval()

print("\n✅ Model loaded and ready!")

In [ ]:
# Step 3: Load test dataset and generate predictions (run ONCE, reused by all cells below)
import json

print("Loading test dataset (data model has NEVER seen)...")
test_dataset = load_dataset("json", data_files={"test": "test.jsonl"})["test"]
print(f"Total test examples available: {len(test_dataset)}")

# Using 20 examples for metrics (change this number if you want more/fewer)
NUM_EVAL_EXAMPLES = 20
test_subset = test_dataset.select(range(min(NUM_EVAL_EXAMPLES, len(test_dataset))))

predictions = []
references  = []
input_codes = []
vuln_types  = []

SYSTEM_PROMPT = """You are an expert Java security auditor. Analyze the provided code.
If the code is secure, output:
\"This Java code is completely secure and contains no vulnerabilities. No changes are required.\"

If the code is vulnerable, output your analysis in this exact format:
### \U0001f6e1\ufe0f Vulnerability Analysis
*   **Status**: VULNERABLE
*   **Type**: [Vulnerability Type]
*   **Severity**: HIGH

### \U0001f4dd Explanation
[Provide a brief explanation of the vulnerability]

### \U0001f6e0\ufe0f Fixed Code
```java
[Fixed complete Java code]
```"""

print(f"\nGenerating model outputs for {NUM_EVAL_EXAMPLES} test examples...")
for i, example in enumerate(test_subset):
    input_code   = example.get('input', '')
    ground_truth = example.get('output', '') or example.get('completion', '')
    vuln_type    = example.get('vulnerability_type', 'Unknown')
    instruction  = example.get('prompt', '') or example.get('instruction', '')

    full_prompt = f"{instruction}\n\n{input_code}" if input_code else instruction
    messages    = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": full_prompt}
    ]

    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=True,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    predictions.append(generated)
    references.append(ground_truth)
    input_codes.append(input_code.strip())
    vuln_types.append(vuln_type)
    print(f"  [{i+1}/{NUM_EVAL_EXAMPLES}] ({vuln_type}) — done")

print("\n✅ All predictions generated! Run any cell below for metrics.")


In [ ]:
# Step 4: Browse Test Examples Side-by-Side
# Change EXAMPLE_INDEX to any number from 0 to NUM_EVAL_EXAMPLES-1 and re-run this cell!

EXAMPLE_INDEX = 0   # <-- CHANGE THIS NUMBER to browse different test examples

is_safe = input_codes[EXAMPLE_INDEX] == references[EXAMPLE_INDEX].strip()
label   = "✅ SAFE CODE" if is_safe else "⚠️ VULNERABLE CODE"

print(f"{'='*60}")
print(f"TEST EXAMPLE #{EXAMPLE_INDEX + 1}  |  {label}  |  Type: {vuln_types[EXAMPLE_INDEX]}")
print(f"{'='*60}")

print(f"\n📥 INPUT CODE (What was given to the model):")
print("-" * 50)
print(input_codes[EXAMPLE_INDEX])

print(f"\n✅ GROUND TRUTH (Expected answer from dataset):")
print("-" * 50)
print(references[EXAMPLE_INDEX][:600])

print(f"\n🤖 MODEL OUTPUT (What the fine-tuned 32B model generated):")
print("-" * 50)
print(predictions[EXAMPLE_INDEX])

In [ ]:
# Step 5: ROUGE + BLEU + BERTScore
import evaluate

rouge     = evaluate.load("rouge")
bleu      = evaluate.load("bleu")
bertscore = evaluate.load("bertscore")

rouge_res = rouge.compute(predictions=predictions, references=references)
bleu_res  = bleu.compute(predictions=predictions, references=[[r] for r in references])
bert_res  = bertscore.compute(predictions=predictions, references=references, lang="en")

print("\n============== STANDARD NLP METRICS ==============")
print(f"ROUGE-1:            {rouge_res['rouge1']:.4f}")
print(f"ROUGE-2:            {rouge_res['rouge2']:.4f}")
print(f"ROUGE-L:            {rouge_res['rougeL']:.4f}")
print(f"BLEU:               {bleu_res['bleu']:.4f}")
print(f"BERTScore Precision:{sum(bert_res['precision'])/len(bert_res['precision']):.4f}")
print(f"BERTScore Recall:   {sum(bert_res['recall'])/len(bert_res['recall']):.4f}")
print(f"BERTScore F1:       {sum(bert_res['f1'])/len(bert_res['f1']):.4f}")
print("==================================================")

In [ ]:
# Step 6: CodeBLEU Score (Gold Standard for Code Generation)
import codebleu.utils
import codebleu.codebleu
import tree_sitter_java

# Runtime Patch: Fix tree-sitter 0.22+ compatibility with codebleu
def patched_get_tree_sitter_language(lang):
    if lang == 'java':
        lang_obj = tree_sitter_java.language()
        if type(lang_obj) == int:
            from tree_sitter import Language
            return Language(lang_obj)
        return lang_obj
    return codebleu.utils._old_get_tree_sitter_language(lang)

if not hasattr(codebleu.utils, '_old_get_tree_sitter_language'):
    codebleu.utils._old_get_tree_sitter_language = codebleu.utils.get_tree_sitter_language
codebleu.codebleu.get_tree_sitter_language = patched_get_tree_sitter_language

from codebleu import calc_codebleu

print("Calculating CodeBLEU (checks AST structure + data flow)...")
result = calc_codebleu(
    references=[[r] for r in references],
    predictions=predictions,
    lang="java",
    weights=(0.25, 0.25, 0.25, 0.25)
)

print("\n============== CODEBLEU SCORE ==============")
print(f"Overall CodeBLEU:        {result['codebleu']:.4f}  ← Main code metric")
print(f"  N-gram Match:          {result['ngram_match_score']:.4f}")
print(f"  Weighted N-gram Match: {result['weighted_ngram_match_score']:.4f}")
print(f"  Syntax Match (AST):    {result['syntax_match_score']:.4f}")
print(f"  Dataflow Match:        {result['dataflow_match_score']:.4f}")
print("============================================")


In [ ]:
# Step 7: Security Confusion Matrix
SAFE_KEYWORDS   = ["completely secure", "no vulnerability", "already safe",
                   "no changes are required", "no vulnerabilities", "is safe", "is secure"]

def is_safe_label(input_code, ground_truth):
    return input_code.strip() == ground_truth.strip()

def model_says_safe(prediction):
    return any(kw in prediction.lower() for kw in SAFE_KEYWORDS)

TP = TN = FP = FN = 0
for pred, ref, inp in zip(predictions, references, input_codes):
    actually_safe     = is_safe_label(inp, ref)
    model_called_safe = model_says_safe(pred)

    if not actually_safe and not model_called_safe:  TP += 1
    elif actually_safe and model_called_safe:         TN += 1
    elif actually_safe and not model_called_safe:     FP += 1
    else:                                             FN += 1

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
accuracy  = (TP + TN) / (TP + TN + FP + FN)

print("\n======== SECURITY CONFUSION MATRIX ========")
print(f"  True Positives  (TP): {TP:3d}  ✅ Vulnerable → correctly flagged")
print(f"  True Negatives  (TN): {TN:3d}  ✅ Safe → correctly identified")
print(f"  False Positives (FP): {FP:3d}  ⚠️  Safe → wrongly flagged")
print(f"  False Negatives (FN): {FN:3d}  ❌ Missed vulnerabilities (CRITICAL!)")
print(f"")
print(f"  Accuracy:  {accuracy*100:.1f}%")
print(f"  Precision: {precision*100:.1f}%")
print(f"  Recall:    {recall*100:.1f}%  ← Most important for a security tool")
print(f"  F1 Score:  {f1*100:.1f}%")
print("============================================")

In [ ]:
# Step 8: Test on ANY custom code you write yourself!
# Change the code below and re-run this cell as many times as you want.

YOUR_CODE = """
public void login(String username, String password) {
    String query = "SELECT * FROM users WHERE username='" + username +
                   "' AND password='" + password + "'";
    ResultSet rs = statement.executeQuery(query);
}
"""

instruction = "Analyze the following Java code. If a vulnerability exists, provide the fixed code. If it is safe, output the original code."
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"{instruction}\n\n{YOUR_CODE}"}
]
text     = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs   = tokenizer(text, return_tensors="pt").to(model.device)

print("📥 Your Input Code:", YOUR_CODE)
print("\n🤖 Model Analysis & Fix:")
print("-" * 50)
with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=512, temperature=0.1,
        do_sample=True, use_cache=True, pad_token_id=tokenizer.eos_token_id
    )
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))